# Probabilistic motion models: notebook

The robot of RO-001 (wheel radius 0.05 m, wheel separation 0.2 m) starts at the pose (2 m, 1 m, 30°) and drives v = 0.5 m/s, ω = 1 rad/s for 1 s. We check every number of the Note: the density of an end pose, wheel odometry from encoder ticks, the odometry and velocity motion models, sampling, and sampling with a map.

In [1]:
import numpy as np
import pandas as pd
from math import atan2, cos, exp, hypot, pi, radians, sin, sqrt

START = (2.0, 1.0, radians(30))

def normal_density(a, b):
    """Density at a of a normal distribution with mean 0 and standard deviation b (Freiburg slide 15)."""
    return exp(-0.5 * a * a / (b * b)) / sqrt(2 * pi * b * b)

## 1. One move, many end positions (Note, Section 2)

In [2]:
for q in (1.0, 0.99, 1.1):
    print(f'density at {q} m: {normal_density(q - 1, 0.1):.3f} per m')
from scipy.stats import norm
print('probability of ending between 1.09 and 1.11 m:', round(norm.cdf(1.11, 1, 0.1) - norm.cdf(1.09, 1, 0.1), 4))

density at 1.0 m: 3.989 per m
density at 0.99 m: 3.970 per m
density at 1.1 m: 2.420 per m


probability of ending between 1.09 and 1.11 m: 0.0484


## 2. Wheel odometry from encoder ticks (Note, Section 3)

In [3]:
r, L, N = 0.05, 0.2, 1000                  # wheel radius, separation (m), ticks per wheel turn
tick = 2 * pi * r / N
dR, dL = 191 * tick, 127 * tick
ds, dth = (dR + dL) / 2, (dR - dL) / L
x, y, th = START
print(f'tick length {tick * 1000:.3f} mm;  dR {dR * 1000:.2f} mm, dL {dL * 1000:.2f} mm')
print(f'forward {ds * 1000:.2f} mm, turn {dth:.4f} rad')
print(f'new pose ({x + ds * cos(th):.4f}, {y + ds * sin(th):.4f}, {th + dth:.4f})')

tick length 0.314 mm;  dR 60.00 mm, dL 39.90 mm
forward 49.95 mm, turn 0.1005 rad
new pose (2.0433, 1.0250, 0.6241)


In [4]:
# A right wheel 1 percent larger than the model thinks; equal tick counts for 10 m
turn = (1.01 * 10 - 10) / L
R = 10.05 / turn
xt, yt = R * sin(turn), R * (1 - cos(turn))
print(f'true turn {turn:.2f} rad, true end ({xt:.2f}, {yt:.2f}), gap to (10, 0): {hypot(xt - 10, yt):.2f} m')

true turn 0.50 rad, true end (9.64, 2.46), gap to (10, 0): 2.49 m


## 3. The velocity model without noise: the exact arc (Note, Section 6.1)

In [5]:
def arc_end(x, y, th, v, w, dt):
    R = v / w
    xc, yc = x - R * sin(th), y + R * cos(th)
    return xc, yc, xc + R * sin(th + w * dt), yc - R * cos(th + w * dt), th + w * dt

xc, yc, xe, ye, te = arc_end(*START, 0.5, 1.0, 1.0)
print(f'turning point ({xc:.3f}, {yc:.3f}); end ({xe:.4f}, {ye:.4f}, {te:.4f} rad = {np.degrees(te):.1f} deg)')
print('one Euler step of 1 s instead:', round(2 + 0.5 * cos(START[2]), 3), round(1 + 0.5 * sin(START[2]), 3))

turning point (1.750, 1.433); end (2.2494, 1.4094, 1.5236 rad = 87.3 deg)
one Euler step of 1 s instead: 2.433 1.25


## 4. The odometry model: turn, drive, turn (Note, Section 4)

In [6]:
ODO_END = (2.249, 1.409, 1.524)

def deltas(p, q):
    rot1 = atan2(q[1] - p[1], q[0] - p[0]) - p[2]
    return rot1, hypot(q[0] - p[0], q[1] - p[1]), q[2] - p[2] - rot1

u = deltas(START, ODO_END)
print('rot1, trans, rot2 =', [round(v, 3) for v in u], '  chord 2R sin(0.5) =', round(2 * 0.5 * sin(0.5), 3))
a1, a2, a3, a4 = 0.1, 0.1, 0.1, 0.01
u = (0.500, 0.479, 0.500)
sd = (a1 * u[0] + a2 * u[1], a3 * u[1] + a4 * (u[0] + u[2]), a1 * u[2] + a2 * u[1])
print('standard deviations:', [round(s, 4) for s in sd])

rot1, trans, rot2 = [0.5, 0.479, 0.5]   chord 2R sin(0.5) = 0.479
standard deviations: [0.0979, 0.0579, 0.0979]


In [7]:
rows = []
for q in [(2.27, 1.38, 1.50), (2.10, 1.60, 1.50)]:
    h = deltas(START, q)
    p = [normal_density(u[i] - h[i], sd[i]) for i in range(3)]
    rows.append(dict(pose=q, rot1_hat=round(h[0], 3), trans_hat=round(h[1], 3), rot2_hat=round(h[2], 3),
                     p1=round(p[0], 3), p2=round(p[1], 3), p3=round(p[2], 4), density=p[0] * p[1] * p[2]))
pd.DataFrame(rows)

,pose,rot1_hat,trans_hat,rot2_hat,p1,p2,p3,density
0,"(2.27, 1.38, 1.5)",0.429,0.466,0.547,3.143,6.723,3.6322,7.674383e+01
1,"(2.1, 1.6, 1.5)",0.882,0.608,0.094,0.002,0.570,0.0008,8.728342e-07


## 5. Sampling (Note, Section 5)

In [8]:
rng = np.random.default_rng(3)
b = 0.1
U = rng.uniform(-b, b, size=(12, 100_000))
print('uniform sd', U[0].std().round(4), ' formula b/sqrt(3) =', round(b / sqrt(3), 4))
print('half the sum of 12: mean', (0.5 * U.sum(0)).mean().round(4), ' sd', (0.5 * U.sum(0)).std().round(4))
print('sqrt(6)/2 x sum of 2: sd', (sqrt(6) / 2 * (U[0] + U[1])).std().round(4))

uniform sd 0.0579  formula b/sqrt(3) = 0.0577
half the sum of 12: mean 0.0001  sd 0.1002
sqrt(6)/2 x sum of 2: sd 0.1


In [9]:
# One sample of the odometry model with chosen noise draws
r1, tr, r2 = 0.500 + 0.05, 0.479 - 0.03, 0.500 - 0.02
x, y, th = START
print(f'heading of the drive {th + r1:.4f} rad; end ({x + tr * cos(th + r1):.4f}, {y + tr * sin(th + r1):.4f}, {th + r1 + r2:.4f})')
# One sample of the velocity model with chosen draws
_, _, xe, ye, te = arc_end(*START, 0.54, 0.9, 1.0)
print(f'velocity sample: R = {0.54 / 0.9:.2f}; end ({xe:.4f}, {ye:.4f}, {te + 0.02:.4f})')

heading of the drive 1.0736 rad; end (2.2142, 1.3946, 1.5536)
velocity sample: R = 0.60; end (2.2935, 1.4316, 1.4436)


In [10]:
import sys
sys.path.insert(0, 'images')
import motion
for g in (False, True):
    xs, ys, ts = motion.sample_velocity(motion.START, np.random.default_rng(0), 2_000_000, gamma=g)
    near = np.hypot(xs - 2.2494, ys - 1.4094) < 0.002
    print(f'gamma {g}: {near.sum()} samples end within 2 mm of (2.249, 1.409); their headings spread {np.degrees(ts[near].std()):.2f} deg')

gamma False: 1659 samples end within 2 mm of (2.249, 1.409); their headings spread 0.24 deg


gamma True: 1659 samples end within 2 mm of (2.249, 1.409); their headings spread 1.71 deg


In [11]:
rng = np.random.default_rng(11)
n, step = 500, (0.1, 0.3, 0.1)
c = tuple(np.full(n, v) for v in motion.START)
spreads = []
for k in range(8):
    c = motion.sample_odometry(step, c, rng, n)
    spreads.append(round(float(np.sqrt(np.var(c[0]) + np.var(c[1]))), 3))
print('spread after 1..8 steps (m):', spreads)

spread after 1..8 steps (m): [0.035, 0.053, 0.076, 0.1, 0.128, 0.157, 0.188, 0.222]


## 6. Sampling with a map (Note, Section 7)

In [12]:
import map_reject
bx, by, _ = map_reject.big_step(np.random.default_rng(5))
sx, sy, _ = map_reject.small_steps(np.random.default_rng(5))
print('one big step: inside the wall', map_reject.inside(bx, by).sum(), ' beyond it', map_reject.beyond(bx, by).sum())
print('twenty small steps: beyond the wall', map_reject.beyond(sx, sy).sum())

one big step: inside the wall 95  beyond it 14
twenty small steps: beyond the wall 0
